# Question 2 — Part 0: Speech Corpus Preparation

Before implementing Persian speech synthesis,
we first need to record a small speech database
using our own voice.

Two corpora are required:

1. Word-level corpus
2. Phoneme-level corpus

All recordings should be:

- Mono
- 16 kHz
- 16-bit PCM
- WAV format

Silence at beginning and end should be minimized.
Consistent speaking speed and tone should be used.

# Question 2 — Part 1: Word-Based Concatenative Number Synthesis

In this section we implement a simple concatenative Text-To-Speech system
for Persian numbers.

Pipeline:

1. Convert an input number into a sequence of Persian number units.

Example:

1405

becomes:

1000 + 400 + ConnectorO + 5

2. Load recorded word audio units from ./words/

3. Concatenate those recorded units in order

4. Generate synthesized speech waveform

5. Save output as a new WAV file

This implements word-level speech synthesis.

In [1]:
# ==========================================
# Part 1 - Word Concatenative TTS
# ==========================================

import os
import numpy as np
import soundfile as sf


WORDS_DIR = "./words"
OUTPUT_DIR = "./outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------
# Persian number decomposition
# ------------------------------------

def number_to_units(n):
    """
    Convert number into
    sequence of recorded units.

    Uses filenames as tokens.
    """

    units = []

    if n == 0:
        return ["0"]

    if n >= 1000000:
        millions = n // 1000000
        units += number_to_units(millions)
        units.append("1000000")
        n %= 1000000

        if n > 0:
            units.append("ConnectorO")


    if n >= 1000:
        thousands = n // 1000
        units += number_to_units(thousands)
        units.append("1000")
        n %= 1000

        if n > 0:
            units.append("ConnectorO")


    hundreds_map = [
        900,800,700,600,
        500,400,300,200,100
    ]

    for h in hundreds_map:

        if n >= h:
            units.append(str(h))
            n -= h

            if n > 0:
                units.append("ConnectorO")
            break


    special_1_to_19 = list(range(1,20))

    if n in special_1_to_19:
        units.append(str(n))
        return units


    tens_map = [
        90,80,70,60,
        50,40,30,20
    ]

    for t in tens_map:

        if n >= t:
            units.append(str(t))
            n -= t

            if n > 0:
                units.append("ConnectorO")
            break


    if n > 0:
        units.append(str(n))

    return units


# ------------------------------------
# Load one word unit
# ------------------------------------

def load_word(token):
    """
    Load one recorded word.
    """

    file_path = os.path.join(
        WORDS_DIR,
        token + ".wav"
    )

    audio, sr = sf.read(
        file_path
    )

    return audio, sr


# ------------------------------------
# Concatenate word units
# ------------------------------------

def synthesize_number(number):

    units = number_to_units(
        number
    )

    print(
        "Units:",
        units
    )

    pieces = []
    sr_final = None

    for token in units:

        audio, sr = load_word(
            token
        )

        if sr_final is None:
            sr_final = sr

        pieces.append(audio)


    synthesized = np.concatenate(
        pieces
    )

    return synthesized, sr_final


# ------------------------------------
# Test required homework numbers
# ------------------------------------

test_numbers = [
    1405,
    202010
]


for n in test_numbers:

    signal, sr = synthesize_number(
        n
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"{n}_word_tts.wav"
    )

    sf.write(
        output_file,
        signal,
        sr
    )

    print(
        f"Saved {output_file}"
    )

Units: ['1', '1000', 'ConnectorO', '400', 'ConnectorO', '5']
Saved ./outputs\1405_word_tts.wav
Units: ['200', 'ConnectorO', '2', '1000', 'ConnectorO', '10']
Saved ./outputs\202010_word_tts.wav


# Question 2 — Part 2: Phoneme-Based Concatenative Synthesis

In this section we synthesize Persian words and phrases
using recorded phoneme units.

Instead of concatenating whole words,
we concatenate phonemes.

Pipeline:

1. Define a phoneme dictionary
mapping words to phoneme sequences.

2. Load phoneme recordings from:

./phonemes/

3. Concatenate phoneme signals
to synthesize words and sentences.

4. Save generated speech files.

This implements a primitive phoneme-based TTS system.

In [3]:
# ==========================================
# Part 2 - Phoneme-Based Synthesis
# ==========================================

import os
import numpy as np
import soundfile as sf

PHONEME_DIR = "./phonemes"
OUTPUT_DIR = "./outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# -----------------------------------
# Example phoneme lexicon
# (adjust to your own recordings)
# -----------------------------------

phoneme_dictionary = {

    # daneshgah
    "دانشگاه":
    [
        "d","aa","n",
        "sh","g","aa","h"
    ],

    # tehran
    "تهران":
    [
        "t","e",
        "h","r","aa","n"
    ],
    # be
    "به":
    [
        "b","e"
    ],

    # kojā
    "کجا":
    [
        "k","o",
        "j","aa"
    ],

    # chenin
    "چنین":
    [
        "ch","e",
        "n","i",
        "n"
    ],

    # shetaban
    "شتابان":
    [
        "sh","e","t",
        "aa","b", 
        "aa","n"
    ],

    # example name
    "فرید":
    [
        "f","a",
        "r","i","d"
    ]
}



# -----------------------------------
# Load phoneme audio
# -----------------------------------

def load_phoneme(ph):
    
    file_path = os.path.join(
        PHONEME_DIR,
        ph + ".wav"
    )

    audio, sr = sf.read(
        file_path
    )

    return audio, sr



# -----------------------------------
# Word synthesis from phonemes
# -----------------------------------

def synthesize_word(word):

    phonemes = phoneme_dictionary[word]

    print(
        word,
        "->",
        phonemes
    )

    parts = []
    sr_final = None

    for ph in phonemes:

        audio, sr = load_phoneme(ph)

        if sr_final is None:
            sr_final = sr

        parts.append(audio)

    synthesized = np.concatenate(
        parts
    )

    return synthesized, sr_final



# -----------------------------------
# Sentence synthesis
# -----------------------------------

def synthesize_phrase(words):

    all_parts = []

    sr_final = None

    for word in words:

        signal, sr = synthesize_word(
            word
        )

        if sr_final is None:
            sr_final = sr

        all_parts.append(signal)

        # tiny pause between words
        pause = np.zeros(
            int(0.03*sr)
        )

        all_parts.append(
            pause
        )


    output = np.concatenate(
        all_parts
    )

    return output, sr_final



# -----------------------------------
# Required homework phrases
# -----------------------------------

phrases = {

"daneshgah_tehran":
[
"دانشگاه",
"تهران"
],

"be_koja_chenin_shetaban":
[
"به",
"کجا",
"چنین",
"شتابان"
],

"my_name":
[
"فرید"
]
}


# Generate outputs
for name, words in phrases.items():

    signal, sr = synthesize_phrase(
        words
    )

    out_file = os.path.join(
        OUTPUT_DIR,
        f"{name}_phoneme.wav"
    )

    sf.write(
        out_file,
        signal,
        sr
    )

    print(
        "Saved:",
        out_file
    )

دانشگاه -> ['d', 'aa', 'n', 'sh', 'g', 'aa', 'h']
تهران -> ['t', 'e', 'h', 'r', 'aa', 'n']
Saved: ./outputs\daneshgah_tehran_phoneme.wav
به -> ['b', 'e']
کجا -> ['k', 'o', 'j', 'aa']
چنین -> ['ch', 'e', 'n', 'i', 'n']
شتابان -> ['sh', 'e', 't', 'aa', 'b', 'aa', 'n']
Saved: ./outputs\be_koja_chenin_shetaban_phoneme.wav
فرید -> ['f', 'a', 'r', 'i', 'd']
Saved: ./outputs\my_name_phoneme.wav
